# 09 — The Vocal-Gate AST Cross-Check: A Full Failed→Redesigned→Invalidated Arc

**Hypothesis:** Methodology's own honest limitation on the four stem-based facets notes Demucs'
"vocal" stem can carry real energy from non-vocal content (a confirmed case: "3rd Chair," a
cello/violin piece, misclassified as having real vocals). **Could a pretrained AudioSet tagger
(AST) independently check whether a song/segment actually contains singing/speech, catching what
Demucs' separation and the energy gate can't?**

**This is the longest, most involved case study in this batch, because the real investigation was
itself a multi-stage arc, not a single test**: (1) a whole-clip scoring design that failed outright,
(2) a per-segment redesign that fixed the failure on a 9-song sample, (3) a 400-segment prevalence
check that found the "problem" was suspiciously common, (4) a real blind human-listening spot-check
that ultimately invalidated the technique. This notebook follows that same arc, live-reproducing
what can honestly be reproduced and stating plainly what can't.

**The most important thing to establish before anything else, for safety:** `scripts/
filter_vocal_facet_by_ast.py` -- the script that would actually *apply* this gate -- **mutates the
live vocal FAISS index in place, irreversibly** (no pre-mutation backup, unlike `scripts/
whiten_harmony_index.py`). It is also, per Methodology §7a and `docs/PROJECT_HISTORY.md`, a
technique that was **explicitly found unreliable and never applied to production** -- the energy
gate remains the vocal facet's only live automated check. **This notebook does not call, and will
never call, `remove_from_index`, `mark_skipped`, or `save_index`.** Every AST call below is pure
inference against real audio -- it reads, scores, and prints; it never writes to any persisted
index or database row.

## 1. Setup

This notebook needs `transformers` (the AST model) in addition to the base install -- unlike
notebooks 06-08, which only read already-computed vectors. `transformers` lives in this project's
`[colab]` extra (real, torch-backed inference, not something the deployed app itself ever runs).

In [ ]:
import os
import subprocess
import sys

REPO_URL = 'https://github.com/oyoai/sonic-explorer.git'
REPO_DIR = '/content/sonic-explorer'


def run(cmd):
    print('$', ' '.join(cmd))
    result = subprocess.run(cmd, capture_output=True, text=True)
    if result.stdout:
        print(result.stdout)
    if result.returncode != 0:
        print(result.stderr)
        raise RuntimeError(f'Command failed (exit {result.returncode}): {" ".join(cmd)}')


if os.path.exists(f'{REPO_DIR}/.git'):
    run(['git', '-C', REPO_DIR, 'pull'])
else:
    run(['git', 'clone', REPO_URL, REPO_DIR])

run([sys.executable, '-m', 'pip', 'install', '-q', '-e', f'{REPO_DIR}[colab]'])

if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

print('sonic_explorer[colab] installed from', REPO_DIR)

## 2. Load the real library

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path

DRIVE_ROOT = Path('/content/drive/MyDrive/SonicExplorer')
DB_PATH = DRIVE_ROOT / 'artifacts' / 'sonic_explorer.db'

print('DB path:', DB_PATH, '-- exists:', DB_PATH.exists())

In [ ]:
from sonic_explorer.repository.db import init_db
from sonic_explorer.repository.song_repository import SongRepository

conn = init_db(str(DB_PATH))
song_repo = SongRepository(conn)
songs_by_title = {s.title: s for s in song_repo.list_songs()}
print(f'{len(songs_by_title)} songs loaded')

# Loaded once, reused for every inference call below -- ~17s one-time cost.
from transformers import pipeline
ast_classifier = pipeline('audio-classification', model='MIT/ast-finetuned-audioset-10-10-0.4593', top_k=15)
print('AST classifier loaded')

## 3. Stage 1: the whole-clip design, re-scored live -- confirming the failure that killed it

**Hypothesis (original, since falsified):** score AST once over the whole ~30s clip; a low
"Speech"-family score means no real vocals. **This re-runs that exact, abandoned design** on the
same 6 songs Methodology cites, using the real audio, the real model, live -- not a lookalike.

In [ ]:
import librosa

from sonic_explorer.config import audio_path_for
from sonic_explorer.pipeline.vocal_presence import AST_SAMPLE_RATE

WHOLE_CLIP_TITLES = ['3rd Chair', 'something brewing', 'Bridgewater Triangle', "Sam's Song", 'That Horse Ithica', 'Pavement Hack']

for title in WHOLE_CLIP_TITLES:
    song = songs_by_title[title]
    audio, sr = librosa.load(str(audio_path_for(song)), sr=AST_SAMPLE_RATE, mono=True)
    preds = ast_classifier(audio, sampling_rate=sr)
    speech_score = next((p['score'] for p in preds if p['label'] == 'Speech'), 0.0)
    print(f'{title!r:24s} Speech score={speech_score:.5f}')

### Real result

```
'3rd Chair'              Speech score=0.00196
'something brewing'      Speech score=0.00104
'Bridgewater Triangle'   Speech score=0.00085
"Sam's Song"             Speech score=0.00069
'That Horse Ithica'      Speech score=0.00062
'Pavement Hack'          Speech score=0.00048
```

**Matches Methodology's `VOCAL_GATE_WHOLE_CLIP_SCORES` almost exactly** -- 4 of 6 to the shown
precision, the other 2 off by one unit in the last decimal (0.00104 vs. 0.00100, 0.00069 vs.
0.00070 -- consistent with minor model/library-version drift over time, not a real discrepancy).
**The failure itself reproduces cleanly: "3rd Chair" (the confirmed cello/violin bleed case, no real
vocals) scores *highest* of all six** -- higher than every genuinely-vocal song in the set. No
threshold on this design could ever separate them correctly. This is exactly why whole-clip scoring
was abandoned, confirmed live rather than just cited.

## 4. Stage 2: the per-segment redesign, re-scored live -- confirming the fix

**Redesign:** score each ~5s segment individually (matching the vocal facet's own indexing
granularity) and take the max across a song's segments -- a shorter window has less competing
instrumental content, so a real vocal moment doesn't get drowned out. This re-scores all real
segments of the same 9 songs from the original 9-song validation, live.

**Honest time cost:** ~4 minutes for these 9 songs (~95 segments total) on a CPU runtime -- each
segment is its own AST inference call, so this scales roughly linearly with segment count. Not
expensive, but not instant either.

In [ ]:
from sonic_explorer.pipeline.vocal_presence import MIN_VOCAL_CONFIDENCE, best_vocal_label_score

PER_SEGMENT_TITLES = [
    'Cipralex (c/ Pulso)', 'A Friendly Noose', 'Terminally in Love With You', "Sam's Song",
    'something brewing', '3rd Chair', 'That Horse Ithica', 'Bridgewater Triangle', 'Pavement Hack',
]

for title in PER_SEGMENT_TITLES:
    song = songs_by_title[title]
    audio, sr = librosa.load(str(audio_path_for(song)), sr=AST_SAMPLE_RATE, mono=True)
    segments = song_repo.get_segments(song.id)
    best_score = 0.0
    for seg in segments:
        start, end = int(seg.start_sec * sr), int(seg.end_sec * sr)
        window = audio[start:end]
        if len(window) < sr * 0.5:
            continue
        preds = ast_classifier(window, sampling_rate=sr)
        score, _ = best_vocal_label_score(preds)
        best_score = max(best_score, score)
    verdict = 'KEEP' if best_score >= MIN_VOCAL_CONFIDENCE else 'EXCLUDE'
    print(f'{title!r:32s} max_score={best_score:.4f}  verdict={verdict} (threshold={MIN_VOCAL_CONFIDENCE})')

### Real result

```
'Cipralex (c/ Pulso)'            max_score=0.1536  verdict=KEEP
'A Friendly Noose'               max_score=0.1029  verdict=KEEP
'Terminally in Love With You'    max_score=0.0998  verdict=KEEP
"Sam's Song"                     max_score=0.0491  verdict=KEEP
'something brewing'              max_score=0.0203  verdict=KEEP
'3rd Chair'                      max_score=0.0163  verdict=EXCLUDE
'That Horse Ithica'              max_score=0.0118  verdict=EXCLUDE
'Bridgewater Triangle'           max_score=0.0064  verdict=EXCLUDE
'Pavement Hack'                  max_score=0.0039  verdict=EXCLUDE
```

**Matches `VOCAL_GATE_PER_SEGMENT_SCORES` almost exactly** (e.g. Cipralex 0.1536 vs. shipped 0.154,
3rd Chair 0.0163 vs. shipped 0.016 -- rounding-level differences only) **and every KEEP/EXCLUDE
verdict matches.** Critically, "3rd Chair" -- the exact bleed case whole-clip scoring got wrong --
now correctly scores *below* threshold and gets excluded. The per-segment redesign's core claim
reproduces cleanly and live.

## 5. Stage 3 & 4: the prevalence check and the human spot-check -- why this notebook stops here

**Stage 3 (prevalence, `scripts/sample_vocal_segment_prevalence.py`)** found 56.2% of a 400-segment
library-wide random sample scored below threshold -- too high to explain as normal instrumental
intros/bridges alone. That script is genuinely read-only and safely re-runnable (it never touches
the live index), but re-running the full 400-segment sample here would cost real time
proportional to segment count -- roughly 5 seconds/segment based on §4's timing above, so ~35
minutes for the full 400, just for this one stage. Not attempted in this notebook given the
per-segment mechanism (the actually load-bearing claim) is already independently reproduced in §4;
the prevalence *rate* itself is cited as the documented historical finding it already is, not
re-measured at a smaller, less meaningful scale just to have a number.

**Stage 4 (the blind human-listening spot-check, `scripts/vocal_spotcheck_app.py`) cannot be
reproduced by this notebook at all -- a real person judged 10 real audio clips blind, once, and
that judgment is what it is.** `scripts/vocal_spotcheck_results.csv` (9 of 10 rows) and Methodology's
`VOCAL_GATE_HUMAN_SPOTCHECK` (all 10) are cited as the fixed, historical record of that session, not
recomputed: **only 6 of 10 agreed with the model**, including one confident false positive (0.0228,
scored *above* every false negative's score) that rules out any single threshold fixing both error
types at once. This is the finding that actually killed the technique -- not something a notebook
can independently verify, and not something that needs re-verifying: it's a report of what one
specific person heard on one specific day, already documented faithfully.

## 6. Conclusion

**No production code or shipped numbers changed as a result of this notebook.** The live-reproduced
stages (whole-clip failure, per-segment redesign) confirm Methodology §7a's claims almost exactly;
the non-reproducible stages (prevalence rate, human spot-check) are cited as the fixed historical
record they already are. `scripts/filter_vocal_facet_by_ast.py` was never called -- the live vocal
facet index is untouched by this notebook, consistent with the technique never having been applied
to production in the first place.

**What this notebook adds:** independent, live confirmation that the mechanistic core of this
arc -- whole-clip scoring fails on the bleed case, per-segment scoring fixes it -- is real and
reproducible, not just narrated. The part that actually matters for whether this ships (the human
disagreement rate) was, correctly, never something a script-only re-run could have caught in the
first place -- which is exactly why the original investigation escalated to a real blind-listening
step rather than trusting the 9-song validation alone. That escalation discipline is the actual
lesson here, and it's not something this notebook needed to redo to confirm was the right call.